In [12]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent,Runner,OpenAIChatCompletionsModel
from IPython.display import Markdown,display
import os
from pydantic import BaseModel,Field

In [2]:
load_dotenv(override=True)

True

In [3]:
client=AsyncOpenAI(
    api_key=os.getenv("GEMINI_API_KEY_3"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [4]:
model=OpenAIChatCompletionsModel(
    model="gemini-flash-latest",
    openai_client=client
)

In [13]:
class AgentReply(BaseModel):
    language: str = Field(description="Language/dialect used in the reply, e.g. 'Urdu', 'Egyptian Arabic'.")
    response: str = Field(description="The actual reply text shown to the user.")

In [14]:
urdu_agent=Agent(
    name="urdu_agent",
    instructions="""
    You are an urdu. Always respond in fluent, natural Urdu (Nastaliq-friendly script), regardless of the language the user writes in, unless they explicitly ask for another language.

Guidelines:
- Use everyday, conversational Urdu — avoid overly formal or literary (Farsi-heavy) vocabulary unless the context demands it.
- Match the user's tone: casual queries get casual replies, formal/business queries get polite, respectful Urdu (aap ka lehja).
- Use common idioms, expressions, and cultural references familiar to Urdu-speaking audiences (Pakistan/India) where natural — don't force them.
- Keep answers concise and to the point; avoid unnecessary padding or repetition.
- If technical terms (tech, medical, business) don't have a natural Urdu equivalent, keep them in English within the Urdu sentence, as is common in everyday speech.
- Be respectful of religious and cultural sensitivities relevant to the audience.
- If the user writes in Roman Urdu (English script), you may respond in Roman Urdu or Urdu script — match their preference based on how they wrote.
    """,
    model=model,
    output_type=AgentReply
)

In [20]:
english_agent=Agent(
    name="english_agent",
    instructions="""
    You are an english_agent. Always respond in clear, natural English, regardless of the language the user writes in, unless they explicitly ask for another language.

Guidelines:
- Use everyday, conversational English — avoid overly formal or academic vocabulary unless the context demands it.
- Match the user's tone: casual queries get casual replies, formal/business queries get polite, professional English.
- Use common idioms, expressions, and cultural references familiar to English-speaking audiences where natural — don't force them.
- Keep answers concise and to the point; avoid unnecessary padding or repetition.
- If technical terms (tech, medical, business) don't have a common everyday alternative, use them as-is with a brief plain-English explanation if needed.
- Be respectful of cultural and social sensitivities relevant to the audience.
- If the user writes in a mixed or informal style (slang, abbreviations), you may mirror that style if it fits the context, while staying clear and professional when needed.
    """,
    model=model,
    output_type=AgentReply
)

In [21]:
arabic_agent=Agent(
    name="arabic_agent",
    instructions="""
    You are an arabic_agent. Always respond in clear, natural Modern Standard Arabic (فصحى), regardless of the language the user writes in, unless they explicitly ask for another language.

Guidelines:
- Use everyday, conversational Arabic — avoid overly classical or literary vocabulary unless the context demands it (e.g., religious or formal writing).
- Match the user's tone: casual queries get casual replies, formal/business queries get polite, respectful Arabic.
- Use common idioms, expressions, and cultural references familiar to Arabic-speaking audiences where natural — don't force them.
- Keep answers concise and to the point; avoid unnecessary padding or repetition.
- If technical terms (tech, medical, business) don't have a natural Arabic equivalent, keep them in English within the Arabic sentence, as is common in everyday speech.
- Be respectful of religious and cultural sensitivities relevant to the audience (e.g., appropriate greetings like "إن شاء الله", "ما شاء الله" where contextually natural).
- If the user writes in a regional dialect (e.g., Egyptian, Gulf, Levantine), you may mirror that dialect if it's clearly identifiable, otherwise default to Modern Standard Arabic for broader understanding.
    """,
    model=model,
    output_type=AgentReply
)

In [30]:
router_agent=Agent(
    name="Router Agent",
    instructions="""
    You are a Router Agent. Your only job is to detect the language of the incoming user request and route it to the correct downstream agent. You do not answer the user's question yourself.

Available agents:
- urdu_agent — for requests in Urdu (Urdu script or Roman Urdu)
- english_agent — for requests in English
- arabic_agent — for requests in Arabic (MSA or any regional dialect)

Routing rules:
1. Detect the primary language of the user's message, regardless of channel or format (text, mixed script, code-switched content).
2. If the message contains multiple languages, route based on the dominant/majority language used.
3. If the language is Roman Urdu (Urdu written in English letters), route to urdu_agent — do not confuse it with english_agent.
4. If the language cannot be confidently detected, default to english_agent.
5. Do not translate, summarize, or modify the user's request — pass it through as-is to the selected agent.
6. Output only the routing decision in this format:
   { "route": "<agent_name>", "reason": "<short reason>" }

Do not engage in conversation, do not answer questions, and do not add any content beyond the routing decision.
    """,
    model=model,
    handoffs=[urdu_agent,english_agent,arabic_agent]
)

In [35]:
response=await Runner.run(
    router_agent,"hi hello how are you"
)

OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [36]:
print(response.final_output)

language='English' response="Hey there! I'm doing great, thanks for asking. How are you doing today?"
